In [ ]:
from sentence_transformers import SentenceTransformer
from src.ingest import load_faq_data
from tqdm.auto import tqdm
import numpy as np
from pprint import pprint

# Motivation

- So far, we have used keyword search with minsearch and sqlitesearch. It matches exact words. If you search for "Docker", the document has to contain "Docker" to come back.
- But look at these two questions:
    - "Can I still join the course after the start date?"
    - "Is it possible to enroll late?"
- They mean the same thing, yet they share almost no words. A keyword engine struggles to match them. We need something that works on meaning, not on the exact words.
- That something is **vector search**. Instead of matching words, it matches ideas.

# The vector search process

We run vector search in two stages.

1.  Offline (indexing): we convert all documents into vectors (arrays of numbers) and store them in an index.
2.  Online (querying): we convert the user's query into a vector with the same model, then find the closest document vectors by similarity.

An embedding model produces these vectors. It's a neural network trained to capture meaning, so texts that mean similar things land on similar vectors. We measure how close two vectors are with a distance metric. The most common one is cosine similarity.

Cosine similarity measures the angle between two vectors:

-   Vectors pointing in the same direction: similarity close to 1 (similar)
-   Vectors at right angles: similarity close to 0 (unrelated)
-   Vectors pointing in opposite directions: similarity close to -1 (opposite meaning)

The larger the cosine similarity, the more similar the two texts are in meaning.

# Keyword search vs vector search

Here's how the two approaches differ:
-   Keyword search matches exact words. Vector search matches meaning.
-   Keyword search suits specific terms, IDs, and names. Vector search suits paraphrased questions and natural language.
-   Keyword search example: "pandas dataframe". Vector search example: "How do I work with tabular data?"
-   Keyword search uses an inverted index (BM25, TF-IDF). Vector search uses a vector index based on cosine similarity.
-   Keyword search misses synonyms and paraphrases. Vector search misses exact term matches.
- Vector search is usually better, but it adds a lot of operational complexity. So never start with vector search. Start with text search, and reach for vectors once you can show they're worth the extra cost.
- In practice the two work best together. [Hybrid search](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/06-best-practices/lessons/02-hybrid-search.md) combines them.

# Building vector search

We'll take the same FAQ dataset from module 1 and build vector search with three tools:

1.  minsearch - in-memory vector search (simplest, good for experiments)
2.  sqlitesearch - persistent vector search backed by SQLite (production-friendly, same API as minsearch)
3.  PGVector - vector search in PostgreSQL (scalable, runs in Docker)

Then we'll plug vector search into our RAG pipeline.

# Embeddings

Before we can do vector search, we need to turn our text into vectors. We call this process embedding: we embed text into a vector space. The vectors we get back are also called "embeddings."

## Word embeddings and sentence embeddings

- This idea comes from [word2vec](https://en.wikipedia.org/wiki/Word2vec). The model learns to place words as points in a multi-dimensional space. Words with similar meanings land close to each other.

- Imagine a 2D space where "enroll" and "join" are near each other and "Docker" is far away.
- The same idea works for entire sentences.
- Now imagine all 1200 documents in our FAQ dataset. Each one becomes a point in this space. When a user asks a question, we embed it into the same space and find the closest documents. Those nearest neighbors are our search results.
- The model encodes the whole sentence, not the words in isolation. So it can tell apart the same word in different contexts.
    - Take the word "judge." In "the judge ruled out the possibility of crime" (legal) it gets one vector. In "LLM-as-a-judge approach to evaluate LLMs" (ML evaluation) it gets a different one. The surrounding context changes the embedding.
- So an embedding model takes text in and returns a fixed-length array of numbers. We train it so that texts with similar meanings get similar vectors.

## Choosing an embedding model

- We'll use [sentence-transformers](https://www.sbert.net/), a popular open-source library for embeddings. It runs locally, so there are no API costs.
- Sentence-transformers supports many models. The right one depends on our task, our language, and the resources you have. Larger models are usually slower
- So for our FAQ dataset of short English texts a small model is enough. We'll use `all-MiniLM-L6-v2`
    -   384-dimensional vectors (compact)
    -   Fast on CPU
    -   Good quality for general English text
    -   Uses cosine similarity (we'll explain this below)

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

## Cosine similarity

The `all-MiniLM-L6-v2` model outputs normalized vectors - vectors with unit length. When both vectors are normalized, the dot product equals cosine similarity. That's why the model documentation says it "uses cosine similarity."

Cosine similarity measures the angle between two vectors, ignoring their length:

-   1.0 = same direction (similar)
-   0.0 = perpendicular (unrelated)
-   \-1.0 = opposite direction (opposite meaning)

Formally, if `theta` is the angle between two vectors, cosine similarity is `cos(theta)`:

-   `cos(0) = 1` - vectors point in the same direction
-   `cos(90) = 0` - vectors are perpendicular
-   `cos(180) = -1` - vectors point in opposite directions

Because our vectors are normalized, the dot product gives us cosine similarity directly. This is why we can use `v1.dot(dv)` to compare texts.

***In practice, we rarely get cosine similarity below 0. The embedding model maps text to a region of the vector space where most vectors have positive components. There's no concept of "opposite meaning" that maps to a vector pointing the other way.***

In [ ]:
# a sample query
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

# a sample doc
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

print(f"similarity: {v1.dot(dv)}")

In [ ]:
# an unrelated query
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)
print(f"similarity: {v2.dot(dv)}")

# Embedding Our Dataset

## Loading the data

In [ ]:
documents = load_faq_data()

## Generating embeddings

Each document is a Python dictionary with a question and an answer. We embed both together. That way a query can match against the question text and the answer text in our index.

In [ ]:
texts = []
for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

Now we generate the embeddings. We have about 1200 texts here. We won't hand the model all of them at once. That takes a long time, and we can't see what's happening inside. Instead we split them into batches.

We turn them into a 2-dimensional array (matrix) where
-   rows are documents (vectors)
-   columns are dimensions of the vectors

In [ ]:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

print(len(vectors))

In [ ]:
X = np.array(vectors)
print(X.shape)

# Vector Search

## Scoring documents

In [ ]:
query = "Can I still join the course after the start date?"
v_query = model.encode(query)
scores = X.dot(v_query)
print(scores.shape)

This is matrix-vector multiplication. Each element `i` of `scores` is the cosine similarity between document `i` (row `i` of `X`) and `v_query`.

## Best match

In [ ]:
idx = np.argmax(scores)
print(idx, scores[idx])
documents[idx]

## Top k results

In [ ]:
k = 5
# topk = np.argsort(scores)[-k:]
# topk = topk[::-1]
topk = np.argpartition(-scores, k)[:k]
print(topk)
print(scores[topk])
print("==============")
for idx in topk:
    print(scores[idx])
    print(documents[idx])
    print()

- This is vector search in its simplest form. We embed the query, compute dot products against all documents, and return the highest-scoring ones.
- We return 5 and not the single best for a reason. The answer to a question can be spread across several documents. One holds part of it, another fills in the rest. Sometimes the top result isn't the right one but the second is. We send all 5 to the LLM and let it combine them.
- The number 5 is a starting point, picked on gut feeling. Later, when we evaluate search quality, we can test whether 3 or 10 works better for our data.

# Vector Search with minsearch

## Creating the index

In [ ]:
from minsearch import VectorSearch
vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

## Searching

In [ ]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)
results = vindex.search(query_vector, num_results=k)
results[0]

In [ ]:
results = vindex.search(
    query_vector,
    filter_dict={"course": "mlops-zoomcamp"},
    num_results=k
)
results[0]